# haql-qc — production data QC with an audit trail


Deterministic data-quality checks for oil & gas production data.


Audits the **Equinor Volve** production dataset, a publicly shared field dataset on Norway's continental shelf (Equinor, 1996–2021).


**Data license:** Volve is published by Equinor for research use. This notebook downloads from a public mirror; check Equinor's terms before commercial use. No Volve data is bundled here.


## 1. Install


Zero runtime dependencies — standard library only.


In [ ]:
# haql-qc is not on PyPI yet. If you are running this notebook from the
# project directory, `sys.path` already has the package -- no install needed.
# Otherwise install the built wheel:  pip install dist/haql_qc-*.whl
import sys, pathlib

sys.path.insert(0, '.')            # prefer the local checkout

import haql_qc
print('haql-qc', haql_qc.__version__, '| ruleset', haql_qc.RULESET_VERSION)
assert haql_qc.__version__ == haql_qc.TOOL_VERSION, 'version drift'
print('import OK')

Runs from a checkout: the project directory is already on `sys.path`, so there is no
`pip install` step and nothing to upload. `haql-qc` is not on PyPI yet; once it is, this
cell becomes `!pip install haql-qc`.

## 2. Fetch the Volve production data


A public mirror of the production workbook. If this fails, upload your own production CSV and continue from cell 3 — the tool is not Volve-specific.


In [ ]:
import urllib.request, os

URL = ('https://raw.githubusercontent.com/yohanesnuwara/'
       'volve-machine-learning/main/Volve%20production%20data.xlsx')
DEST = 'Volve_production_data.xlsx'

if not os.path.exists(DEST):
    urllib.request.urlretrieve(URL, DEST)
print(DEST, os.path.getsize(DEST), 'bytes')

import hashlib
print('sha256:', hashlib.sha256(open(DEST,'rb').read()).hexdigest())

In [ ]:
# Read the workbook. openpyxl is only needed for this step; the haql-qc
# audit itself needs nothing outside the standard library.
import importlib.util, subprocess, sys

if importlib.util.find_spec('openpyxl') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], check=True)

import openpyxl
book = openpyxl.load_workbook(DEST, read_only=True, data_only=True)
print('sheets:', book.sheetnames)          # note: openpyxl spells it sheetnames
for name in book.sheetnames:
    sheet = book[name]
    print(f'  {name}: {sheet.max_row} rows x {sheet.max_column} cols')

Convert the `Daily Production Data` sheet to CSV. `haql-qc` reads CSV, and keeping the conversion explicit means the audited bytes are exactly the ones you can hash and re-derive — column order and cell values are preserved byte-for-byte from the sheet.


In [ ]:
import openpyxl, csv

# The daily sheet has a clean header on row 1 and 15,634 data rows.
wb = openpyxl.load_workbook(DEST, read_only=True, data_only=True)
ws = wb['Daily Production Data']
records = ws.iter_rows(values_only=True)
header = list(next(records))

with open('volve_daily.csv', 'w', newline='') as fh:
    out = csv.writer(fh)
    out.writerow(header)
    for rec in records:
        if rec[0] is None:      # trailing blank rows
            continue
        out.writerow(['' if v is None else v for v in rec])

print('wrote volve_daily.csv')
print('sha256:', hashlib.sha256(open('volve_daily.csv','rb').read()).hexdigest())

## 3. Run the audit


Volve reports production in Sm³ with no unit in the column name, and gives daily on-stream hours rather than a day count. The unit must be declared — the tool will not infer it from the word "oil".


In [ ]:
VOLVE_UNITS = {
    'BORE_OIL_VOL': 'Sm3',
    'BORE_GAS_VOL': 'Sm3',
    'BORE_WAT_VOL': 'Sm3',
    'BORE_WI_VOL': 'Sm3',
}
print(VOLVE_UNITS)

In [ ]:
from haql_qc import load_production_csv, run_all

rows, columns, report = load_production_csv(
    'volve_daily.csv',
    well_column='WELL_BORE_CODE',
    date_column='DATEPRD',
)
print('rows', report.rows_read, '| wells', report.wells_seen)
print('columns:', list(columns)[:12])
print('inferred cadence:', report.config.get('cadence_mode'))

In [ ]:
report.extend(run_all(rows, columns, unit_declarations=VOLVE_UNITS))
import json
print(json.dumps(report.summary(), indent=2))

## 4. What did it find?


In [ ]:
from collections import Counter

counts = Counter(i.rule_id for i in report.issues)
for rule, n in counts.most_common():
    print(f'{rule:<28} {n:>4}')

On the real Volve daily export (15,634 rows, 7 wells) the tool reports 42 findings: 38 reporting-cadence gaps and **4 negative water rates**.


The negative water rates are the interesting result. An earlier version of this tool reported **zero** findings on that column, because it did not recognise `BORE_WAT_VOL` as produced water at all. Two of the readings are serious:


In [ ]:
neg = [i for i in report.issues if i.rule_id == 'QC010_NEGATIVE_RATE']
for i in neg:
    print(f"row {i.source_row:>6}  {i.well_id:<8} {i.date}  {i.observed:>10} {i.unit}")

## 5. Every finding is traceable by hand


Each issue carries a 1-indexed source row and column, so you can open the CSV and check the claim yourself. No trust required.


In [ ]:
print(neg[0].to_json_line()[:600])

import csv
row_no = neg[0].source_row
with open('volve_daily.csv', newline='') as fh:
    raw = list(csv.reader(fh))
header = raw[0]
vals = raw[row_no - 1]        # source_row is 1-indexed, header included
idx = header.index(neg[0].column)
print('\nheader cell :', header[idx])
print('raw cell    :', repr(vals[idx]))
print('tool claim  :', neg[0].observed, neg[0].unit)

## 6. Skipped checks are visible, not silent


If a rate column has no declared unit, the plausibility check is skipped — and the tool says so with an INFO finding. A skipped check never looks like a clean one.


In [ ]:
rows, columns, bare = load_production_csv(
    'volve_daily.csv', well_column='WELL_BORE_CODE', date_column='DATEPRD')
bare.extend(run_all(rows, columns, unit_declarations={}))   # declare nothing
for i in bare.issues:
    if i.rule_id == 'QC015_IMPLAUSIBLE_RATE':
        print(i.severity.value.upper(), '|', i.message)

## 7. Write the audit report


The report embeds the input's SHA-256, both version numbers, the full runtime config, and per-rule counts. Someone who did not run the tool can check it.


In [ ]:
with open('audit.json', 'w') as fh:
    fh.write(report.to_json())
print('wrote audit.json')

meta = json.load(open('audit.json'))
print('tool          :', meta['tool'], meta['tool_version'])
print('ruleset       :', meta['ruleset_version'])
print('input sha256  :', meta['input']['sha256'])
print('rows audited  :', meta['input']['rows_read'])

In [ ]:
# jsonl for jq / log shipping: one header record, then one line per issue
with open('audit.jsonl', 'w') as fh:
    fh.write(report.to_jsonl())
print('wrote audit.jsonl')
!head -c 400 audit.jsonl

## 8. Use it as a CLI gate in CI


Exit codes: `0` clean at the chosen threshold, `1` a finding met or exceeded it, `2` bad invocation. `--fail-on` defaults to `never` so a report never breaks a pipeline by surprise.


In [ ]:
# Run through `python -m` so this works whether or not the console script is on PATH.
import subprocess, sys

argv = [
    sys.executable, '-m', 'haql_qc.cli',
    '-i', 'volve_daily.csv',
    '--well-column', 'WELL_BORE_CODE', '--date-column', 'DATEPRD',
    '--declare-unit', 'BORE_OIL_VOL=Sm3', '--declare-unit', 'BORE_GAS_VOL=Sm3',
    '--declare-unit', 'BORE_WAT_VOL=Sm3', '--declare-unit', 'BORE_WI_VOL=Sm3',
    '--fail-on', 'high', '-r', 'ci_audit.json',
]
rc = subprocess.run(argv).returncode
print('exit code:', rc)
print(' 0 = no finding at/above high, 1 = one did, 2 = bad invocation')

## 9. Design stance


Three rules, in priority order:


1. **Never corrupt source data.** The tool reads and reports. Corrections, when they exist, are opt-in and written to a separate file.


2. **Never guess.** A unit, a column meaning, or a reporting cadence is not inferred when inference could put a false claim in the audit record. When the tool cannot establish something, it says so rather than reporting "clean".


3. **Never be untraceable.** Every finding carries a stable `rule_id`, a severity, a 1-indexed `source_row`, the observed value, and the input's SHA-256.


No LLM call, no network access, no randomness. The same bytes always produce the same report.
